In [2]:
from pathlib import Path
import pandas as pd
import joblib

ROOT = Path("..")
OUT = ROOT / "powerbi"
OUT.mkdir(exist_ok=True)

df = pd.read_csv(ROOT / "data/processed/jugadores_clean.csv")
arte = joblib.load(ROOT / "models/modelado.joblib")
pipe = arte["pipe_logit"]

METRICAS = ["PrecisionPase", "VelReaccion", "Efectividad1v1", "VelSprint",
            "DistAltaInt", "AciertoRegate", "Recuperaciones", "TomaDecision"]

# --- Tabla de dimensión: jugadores ---
dim_jugadores = df[["Jugador", "Posicion", "Edad"]].copy()

# --- Tabla de hechos: métricas en formato largo ---
fact_rendimiento = df.melt(id_vars="Jugador", value_vars=METRICAS,
                           var_name="MetricaID", value_name="Valor")

# --- Tabla de dimensión: catálogo de métricas ---
INFO = {
    "PrecisionPase":  ("Precisión de pase", "%",      "Técnica",   "Mayor es mejor"),
    "VelReaccion":    ("Velocidad de reacción", "s",  "Cognitiva", "Menor es mejor"),
    "Efectividad1v1": ("Efectividad 1v1", "%",        "Técnica",   "Mayor es mejor"),
    "VelSprint":      ("Velocidad de sprint", "km/h", "Física",    "Mayor es mejor"),
    "DistAltaInt":    ("Dist. alta intensidad", "x100 m/90'", "Física", "Mayor es mejor"),
    "AciertoRegate":  ("Acierto en regate", "%",      "Técnica",   "Mayor es mejor"),
    "Recuperaciones": ("Recuperaciones", "por 90'",   "Táctica",   "Mayor es mejor"),
    "TomaDecision":   ("Toma de decisiones", "0-10",  "Cognitiva", "Mayor es mejor"),
}
dim_metricas = pd.DataFrame(
    [(k, *v) for k, v in INFO.items()],
    columns=["MetricaID", "Metrica", "Unidad", "Dimension", "Direccion"])

# --- Tabla de predicciones del modelo ---
predicciones = df[["Jugador"]].copy()
predicciones["ProbPotencial"] = pipe.predict_proba(df[METRICAS])[:, 1].round(3)
predicciones["AltoPotencial"] = df["AltoPotencial"]

# Exportar en formato "CSV español" (separador ; y decimal ,) para que
# Power BI en Windows con configuración regional española los lea sin fricciones
for nombre, tabla in [("dim_jugadores", dim_jugadores),
                      ("dim_metricas", dim_metricas),
                      ("fact_rendimiento", fact_rendimiento),
                      ("predicciones", predicciones)]:
    tabla.to_csv(OUT / f"{nombre}.csv", index=False, sep=";", decimal=",")
    print(nombre, tabla.shape)

dim_jugadores (40, 3)
dim_metricas (8, 5)
fact_rendimiento (320, 3)
predicciones (40, 3)
